In [ ]:
import sys
import os

# Add the project root to the path
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Import our custom modules
from src import config
from src import data_processing as dp
from src import eda
from src import models
from src import evaluation

# Other imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display settings
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

# Setup Altair for visualizations
eda.setup_altair()

print(" All modules loaded successfully!")

import importlib

importlib.reload(config)
importlib.reload(dp)

print("✓ Modules reloaded")
print(f"✓ Config has DIABETIC_DATA_CSV: {hasattr(config, 'DIABETIC_DATA_CSV')}")
if hasattr(config, "DIABETIC_DATA_CSV"):
    print(f"  Path: {config.DIABETIC_DATA_CSV}")
    print(f"  Exists: {os.path.exists(config.DIABETIC_DATA_CSV)}")

X_train, X_val, X_test, Y_train, Y_val, Y_test, raw_df = dp.preprocess_pipeline()

print("\n✓ Data preprocessing complete!")
print(f"\nDataset shapes:")
print(f"  Train: X={X_train.shape}, Y={Y_train.shape}")
print(f"  Val:   X={X_val.shape}, Y={Y_val.shape}")
print(f"  Test:  X={X_test.shape}, Y={Y_test.shape}")

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

: 

In [3]:
x_train =X_train.drop(columns=['Unnamed: 0'])
x_val =X_val.drop(columns=['Unnamed: 0'])
x_test =X_test.drop(columns=['Unnamed: 0'])

y_train = Y_train['readmitted'].to_numpy()
y_val = Y_val['readmitted'].to_numpy()
y_test = Y_test['readmitted'].to_numpy()

print('Shape of X_train: ', x_train.shape)
print('Shape of X_val: ', x_val.shape)
print('Shape of X_test: ', x_test.shape)
print('Shape of Y_train: ', y_train.shape)
print('Shape of Y_val: ', y_val.shape)
print('Shape of Y_test: ', y_test.shape)


Shape of X_train:  (42910, 42)
Shape of X_val:  (14304, 42)
Shape of X_test:  (14304, 42)
Shape of Y_train:  (42910,)
Shape of Y_val:  (14304,)
Shape of Y_test:  (14304,)


# Bagging

In [4]:
# random forest
forest = RandomForestClassifier(criterion = 'entropy', n_estimators=20, max_depth = 10, min_samples_split=3)
rf_clf = forest.fit(x_train, y_train)

train_accuracy = rf_clf.score(x_train, y_train)

print(f"Train accuracy is {train_accuracy*100:.2f}%.")

val_accuracy = rf_clf.score(x_val, y_val)

print(f"Validation accuracy is {val_accuracy*100:.2f}%.")

test_accuracy = rf_clf.score(x_test, y_test)

print(f"Test accuracy is {test_accuracy*100:.2f}%.")

Train accuracy is 65.58%.
Validation accuracy is 63.26%.
Test accuracy is 62.24%.


In [5]:
# all features
dt = DecisionTreeClassifier(criterion='entropy', max_depth=10)
bag = BaggingClassifier(estimator=dt, n_estimators=100)
b_clf = bag.fit(x_train, y_train)

train_accuracy = b_clf.score(x_train, y_train)

print(f"Train accuracy is {train_accuracy*100:.2f}%.")

val_accuracy = b_clf.score(x_val, y_val)

print(f"Validation accuracy is {val_accuracy*100:.2f}%.")

test_accuracy = b_clf.score(x_test, y_test)

print(f"Test accuracy is {test_accuracy*100:.2f}%.")

Train accuracy is 66.96%.
Validation accuracy is 63.24%.
Test accuracy is 62.76%.


# Boosting

In [6]:
# gradient boosting
gb = GradientBoostingClassifier(criterion='squared_error', n_estimators=500)
gb_clf = gb.fit(x_train, y_train)

train_accuracy = gb_clf.score(x_train, y_train)

print(f"Train accuracy is {train_accuracy*100:.2f}%.")

val_accuracy = gb_clf.score(x_val, y_val)

print(f"Validation accuracy is {val_accuracy*100:.2f}%.")

test_accuracy = gb_clf.score(x_test, y_test)

print(f"Test accuracy is {test_accuracy*100:.2f}%.")

Train accuracy is 64.72%.
Validation accuracy is 63.17%.
Test accuracy is 62.96%.


In [7]:
# ada boosting
ada_tree = DecisionTreeClassifier(criterion='entropy', max_depth=1)
ada = AdaBoostClassifier(estimator=ada_tree, n_estimators=500, learning_rate=0.01)
ada_clf = ada.fit(x_train, y_train)

train_accuracy = ada_clf.score(x_train, y_train)

print(f"Train accuracy is {train_accuracy*100:.2f}%.")

val_accuracy = ada_clf.score(x_val, y_val)

print(f"Validation accuracy is {val_accuracy*100:.2f}%.")

test_accuracy = ada_clf.score(x_test, y_test)

print(f"Test accuracy is {test_accuracy*100:.2f}%.")

Train accuracy is 61.50%.
Validation accuracy is 62.22%.
Test accuracy is 61.63%.


# Neural Network

In [8]:
def build_model(learning_rate=0.01):

  tf.keras.backend.clear_session()

  model = tf.keras.Sequential()

  model.add(keras.Input(shape=(x_train.shape[1],)))

  model.add(keras.layers.Dense(
      units=64,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=32,
      activation='relu'
  ))

  model.add(keras.layers.BatchNormalization())
  model.add(keras.layers.Dropout(0.2))


  model.add(keras.layers.Dense(
      units=16,
      activation='relu'
  ))

  # model.add(keras.layers.BatchNormalization())
  # model.add(keras.layers.Dropout(0.2))

  model.add(keras.layers.Dense(
      units=1,
      activation='sigmoid',
      kernel_initializer='glorot_uniform',
      bias_initializer='glorot_uniform'
  ))

  model.compile(
      optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
      loss=keras.losses.BinaryCrossentropy(),
      metrics=['accuracy']
  )

  history = model.fit(
      x=x_train,
      y=y_train,
      validation_data=(x_val, y_val),
      batch_size=64,
      epochs=10,
      verbose=1
  )

  return model, history

In [9]:
m1, history = build_model(0.001)

Epoch 1/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5628 - loss: 0.7137 - val_accuracy: 0.6000 - val_loss: 0.6679
Epoch 2/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.5933 - loss: 0.6726 - val_accuracy: 0.6030 - val_loss: 0.6655
Epoch 3/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6040 - loss: 0.6643 - val_accuracy: 0.6104 - val_loss: 0.6602
Epoch 4/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6024 - loss: 0.6629 - val_accuracy: 0.6165 - val_loss: 0.6559
Epoch 5/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6142 - loss: 0.6567 - val_accuracy: 0.6226 - val_loss: 0.6530
Epoch 6/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6162 - loss: 0.6544 - val_accuracy: 0.6238 - val_loss: 0.6528
Epoch 7/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6242 - loss: 0.6533 - val_accuracy: 0.6210 - val_loss: 0.6509
Epoch 8/10
671/671 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6155 - loss: 0.6545 - val_accuracy: 0.

In [10]:
preds = (m1.predict(x_val) > 0.5).astype(int)
print(np.unique(preds, return_counts=True))

447/447 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
(array([0, 1]), array([12719,  1585]))
